# SpaceX API data collection

Original capstone code. API v4 launches are enriched with rockets, payloads, cores and launchpads; Falcon 9 single-payload launches through 2020-11-13 are retained.

The live endpoint returned an HTTP service error during this project. The analysis therefore uses the official IBM course frozen snapshot, rather than claiming a successful live collection. Current API data may differ from the course sample.

GitHub: https://github.com/giusdp45/spacex-landing-capstone

In [1]:
"""Collect Falcon 9 records with SpaceX API v4; preserve a documented frozen fallback."""
from pathlib import Path
import json
import requests
import pandas as pd

ROOT = Path.cwd().parent
API = "https://api.spacexdata.com/v4"


def collect_live():
    session = requests.Session()
    cache = {}
    def get(path):
        if path not in cache:
            response = session.get(f"{API}/{path}", timeout=40)
            response.raise_for_status()
            cache[path] = response.json()
        return cache[path]
    records = []
    for launch in get("launches/past"):
        if len(launch.get("payloads", [])) != 1 or len(launch.get("cores", [])) != 1:
            continue
        rocket = get(f"rockets/{launch['rocket']}")
        if rocket["name"] != "Falcon 9" or launch["date_utc"][:10] > "2020-11-13":
            continue
        payload = get(f"payloads/{launch['payloads'][0]}")
        pad = get(f"launchpads/{launch['launchpad']}")
        flight_core = launch["cores"][0]
        if not flight_core.get("core"):
            continue
        core = get(f"cores/{flight_core['core']}")
        outcome = f"{flight_core.get('landing_success')} {flight_core.get('landing_type')}"
        records.append(dict(FlightNumber=launch["flight_number"], Date=launch["date_utc"][:10],
                            BoosterVersion=rocket["name"], PayloadMass=payload.get("mass_kg"),
                            Orbit=payload.get("orbit"), LaunchSite=pad["name"], Outcome=outcome,
                            Flights=flight_core.get("flight"), GridFins=flight_core.get("gridfins"),
                            Reused=flight_core.get("reused"), Legs=flight_core.get("legs"),
                            LandingPad=flight_core.get("landpad"), Block=core.get("block"),
                            ReusedCount=core.get("reuse_count"), Serial=core.get("serial"),
                            Longitude=pad["longitude"], Latitude=pad["latitude"]))
    return pd.DataFrame(records)


def collect_or_fallback():
    try:
        frame = collect_live()
        frame.to_csv(ROOT / "data" / "api_live.csv", index=False)
        status = {"mode": "live", "rows": len(frame), "source": API}
    except requests.RequestException as exc:
        frame = pd.read_csv(ROOT / "data" / "dataset_part_1.csv")
        status = {"mode": "official course snapshot", "rows": len(frame), "live_error": str(exc),
                  "source": "IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv"}
    (ROOT / "data" / "api_collection_status.json").write_text(json.dumps(status, indent=2), encoding="utf-8")
    return frame, status




In [2]:
frame, status = collect_or_fallback()
print(json.dumps(status, indent=2))
display(frame.head())
print(frame.shape)

{
  "mode": "official course snapshot",
  "rows": 90,
  "live_error": "525 Server Error: <none> for url: https://api.spacexdata.com/v4/launches/past",
  "source": "IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv"
}


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


(90, 17)


Source: https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv

API: https://github.com/r-spacex/SpaceX-API/tree/master/docs/v4